# Module 4: Session Managers (10 min)

Add file-based persistence to the customer service agent. Stop the agent, restart it, and watch it remember the previous conversation.

**Prerequisites:** Modules 1-3 completed

In [ ]:
!pip install -q -r requirements.txt

---

## Part 1: Agent Without Persistence (The Problem)

By default, agents lose their memory when you recreate them.

In [ ]:
from strands import Agent, AgentSkills
from strands.agent.conversation_manager import SlidingWindowConversationManager
from customer_service_tools import lookup_customer, get_order_history, process_refund

SYSTEM_PROMPT = """You are a customer service agent for an online electronics store.
Be helpful, professional, and concise.

If there are previous messages in the conversation history, use that context
to continue helping the customer without asking them to repeat information."""

# First interaction
agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    system_prompt=SYSTEM_PROMPT,
)
agent("Hi, I'm customer C-1001. Can you look up my account?")

print(f"\n📝 Messages stored: {len(agent.messages)}")

In [ ]:
# Simulate a "restart" — create a new agent instance
agent2 = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    system_prompt=SYSTEM_PROMPT,
)

print(f"Messages after restart: {len(agent2.messages)}")
# The agent has no memory of the previous conversation!
agent2("What was my account status again?")

---

## Part 2: Add FileSessionManager

The `FileSessionManager` saves conversation history to disk. On restart, it reloads the messages.

In [ ]:
from strands.session.file_session_manager import FileSessionManager

# Clean up any previous session files
import shutil, os
if os.path.exists("./sessions"):
    shutil.rmtree("./sessions")

session_manager = FileSessionManager(
    session_id="customer-session-001",
    storage_dir="./sessions",
)

agent = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    plugins=[AgentSkills(skills=["./skills"])],
    system_prompt=SYSTEM_PROMPT,
    conversation_manager=SlidingWindowConversationManager(window_size=20),
    session_manager=session_manager,
)

# First interaction — this gets saved to disk
agent("Hi, I'm customer C-1001. Can you look up my account?")
print(f"\n📁 Session saved. Messages: {len(agent.messages)}")

---

## Part 3: Restart and Remember

Create a brand new agent with the same session ID. It should remember everything.

In [ ]:
# Simulate restart — new agent, same session_id
session_manager_2 = FileSessionManager(
    session_id="customer-session-001",
    storage_dir="./sessions",
)

agent_restarted = Agent(
    tools=[lookup_customer, get_order_history, process_refund],
    plugins=[AgentSkills(skills=["./skills"])],
    system_prompt=SYSTEM_PROMPT,
    conversation_manager=SlidingWindowConversationManager(window_size=20),
    session_manager=session_manager_2,
)

print(f"🔄 Restored messages: {len(agent_restarted.messages)}")
print("The agent remembers the previous conversation!\n")

# Ask a follow-up — the agent should know we're C-1001
agent_restarted("What orders do I have? You should already know my customer ID.")

---

## 🎯 Try It Yourself

Check what's stored on disk:

In [ ]:
import json

# FileSessionManager stores data in nested folders:
#   ./sessions/session_<id>/session.json
#   ./sessions/session_<id>/agents/agent_<id>/agent.json
#   ./sessions/session_<id>/agents/agent_<id>/messages/message_*.json
# So walk the tree recursively instead of listing only the top level.
session_dir = "./sessions"
for root, _dirs, files in os.walk(session_dir):
    for f in sorted(files):
        if f.endswith(".json"):
            filepath = os.path.join(root, f)
            rel = os.path.relpath(filepath, session_dir)
            print(f"📄 {rel} ({os.path.getsize(filepath)} bytes)")

# Count how many message files were persisted
message_files = [
    os.path.join(root, f)
    for root, _dirs, files in os.walk(session_dir)
    for f in files
    if f.startswith("message_") and f.endswith(".json")
]
print(f"\n💬 Messages persisted to disk: {len(message_files)}")

---

## What's Next

The agent is persistent and follows rules. But what happens when a customer has a technical issue the agent can't solve? In **Module 5: Multi-Agent**, you'll add a tech support specialist that the agent can delegate to.